In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import os
from torchvision import transforms, models
import torch.nn as nn
import torch.optim as optim
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 0.001
EPOCHS = 10
BATCH_SIZE = 32

class ChessDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, is_test=False):
        super().__init__()
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

        if not is_test:
            # Label encoding
            self.label_map = {label: i for i, label in enumerate(self.data['label'].unique())}
            self.inv_label_map = {i: label for label, i in self.label_map.items()}
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        img_name = os.path.join(self.img_dir, self.data.iloc[idx]['image_path'])
        image = Image.open(img_name).convert('RGB')

        if self.transform:
            image = self.transform(image)
            
        if self.is_test:
            return image, self.data.iloc[idx]['id']
            
        label = self.label_map[self.data.iloc[idx]['label']]
        return image, label

# Data augumentation
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True # Freeze pretrained layers at first

num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5) # 5 pieces
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /home/kraz09/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 110MB/s] 


In [ ]:
train_ds = ChessDataset(csv_file='train.csv', img_dir='images/', transform=train_transforms)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_ds = ChessDataset(csv_file='test.csv', img_dir='images/', transform=test_transforms, is_test=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

def train_model(model, train_loader, criterion, optimizer, epochs):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct_preds = 0
        total_preds = 0

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct_preds += (predicted == labels).sum().item()
            total_preds += labels.size(0)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = (correct_preds / total_preds) * 100

        print(f"Epoch [{epoch + 1}/{epochs}] - Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.2f}%")

In [ ]:
train_model(model, train_loader, criterion, optimizer, EPOCHS)

Epoch [1/10] - Loss: 1.0344, Acc: 61.73%
Epoch [2/10] - Loss: 0.5355, Acc: 83.85%
Epoch [3/10] - Loss: 0.3721, Acc: 87.12%
Epoch [4/10] - Loss: 0.4625, Acc: 86.92%
Epoch [5/10] - Loss: 0.3057, Acc: 91.54%
Epoch [6/10] - Loss: 0.2513, Acc: 92.12%
Epoch [7/10] - Loss: 0.2433, Acc: 92.50%
Epoch [8/10] - Loss: 0.1705, Acc: 95.19%
Epoch [9/10] - Loss: 0.0988, Acc: 97.12%
Epoch [10/10] - Loss: 0.1446, Acc: 95.19%


In [16]:
def generate_submission(model, test_loader, inv_label_map):
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for images, ids in test_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            
            for i in range(len(preds)):
                predictions.append({
                    'id': ids[i],
                    'label': inv_label_map[preds[i].item()]
                })
    
    df_sub = pd.DataFrame(predictions)
    df_sub.to_csv('submission.csv', index=False)
inv_label_map = train_ds.inv_label_map
generate_submission(model, test_loader, inv_label_map)